In [1]:
import os
from pathlib import Path
import pandas as pd
import gc


BASE_PATH = Path.cwd().parents[0]
GOLD_PATH = BASE_PATH / "data" / "gold"

GOLD_PATH.mkdir(parents=True, exist_ok=True)

In [2]:
df=pd.read_csv(GOLD_PATH / "la_port_ml_ready_dataset.csv")

In [3]:
df.shape

(233520, 14)

In [4]:
df.head()

,BerthTime,AvgSOG,VesselDraft,ArrivalHour,ArrivalDayOfWeek,TerminalID,BerthID,DailyCapacityTEU,DraftMargin,LOAMargin,DelayMinutes,IsDelayed,BerthFeasible,CongestionLevel
0,232.71,0.051193,6.7,13,6,T1,B1,15298,9.2,120.0,21083.23,1,1,High
1,232.71,0.051193,6.7,13,6,T1,B1,15298,9.2,120.0,21083.23,1,1,High
2,232.71,0.051193,6.7,13,6,T1,B1,15298,9.2,120.0,21083.23,1,1,High
3,232.71,0.051193,6.7,13,6,T1,B2,18334,8.8,198.0,21083.23,1,1,High
4,232.71,0.051193,6.7,13,6,T1,B2,18334,8.8,198.0,21083.23,1,1,High


In [5]:
df.columns

Index(['BerthTime', 'AvgSOG', 'VesselDraft', 'ArrivalHour', 'ArrivalDayOfWeek',
       'TerminalID', 'BerthID', 'DailyCapacityTEU', 'DraftMargin', 'LOAMargin',
       'DelayMinutes', 'IsDelayed', 'BerthFeasible', 'CongestionLevel'],
      dtype='object')

In [6]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


In [7]:
#✅ INPUT FEATURES (used for regression)

In [8]:
FEATURES = [
    "ArrivalHour",
    "ArrivalDayOfWeek",
    "BerthFeasible",
    "BerthTime"
]

X = df[FEATURES]
y = df["DelayMinutes"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


In [9]:
def evaluate(model, X_test, y_test):
    preds = model.predict(X_test)
    print("MAE :", mean_absolute_error(y_test, preds))
    print("RMSE:", mean_squared_error(y_test, preds))
    print("R2  :", r2_score(y_test, preds))


## Linear Regression (Baseline)

In [10]:
from sklearn.linear_model import LinearRegression

lr = LinearRegression()
lr.fit(X_train, y_train)

print("Linear Regression")
evaluate(lr, X_test, y_test)


Linear Regression
MAE : 65542.51232395724
RMSE: 10675960658.333109
R2  : 0.3023419379264085


## Random Forest Regressor

In [11]:
from sklearn.ensemble import RandomForestRegressor

rf = RandomForestRegressor(
    n_estimators=300,
    max_depth=18,
    min_samples_leaf=5,
    n_jobs=-1,
    random_state=42
)

rf.fit(X_train, y_train)

print("Random Forest")
evaluate(rf, X_test, y_test)


Random Forest
MAE : 976.298116569347
RMSE: 46473936.46699976
R2  : 0.9969629977582214


## XGBRegressor

In [12]:
from xgboost import XGBRegressor

xgb = XGBRegressor(
    n_estimators=400,
    max_depth=8,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="reg:squarederror",
    random_state=42,
    n_jobs=-1
)

xgb.fit(X_train, y_train)

print("XGBoost")
evaluate(xgb, X_test, y_test)


XGBoost
MAE : 7855.32058741811
RMSE: 355359595.32178026
R2  : 0.9767777819209238


## GradientBoostingRegressor

In [13]:
from sklearn.ensemble import GradientBoostingRegressor

gbr = GradientBoostingRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=5,
    random_state=42
)

gbr.fit(X_train, y_train)

print("Gradient Boosting")
evaluate(gbr, X_test, y_test)


Gradient Boosting
MAE : 24251.415904683534
RMSE: 1454756170.3702655
R2  : 0.9049338599971402


## LGBMRegressor

In [14]:
from lightgbm import LGBMRegressor

lgbm = LGBMRegressor(
    n_estimators=500,
    learning_rate=0.05,
    num_leaves=31,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

lgbm.fit(X_train, y_train)

print("LightGBM")
evaluate(lgbm, X_test, y_test)


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.008124 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 288
[LightGBM] [Info] Number of data points in the train set: 186816, number of used features: 4
[LightGBM] [Info] Start training from score 63383.860749
LightGBM
MAE : 15359.886575013545
RMSE: 588005399.4169444
R2  : 0.961574726567971


## AdaBoostRegressor

In [15]:
from sklearn.ensemble import AdaBoostRegressor

ada = AdaBoostRegressor(
    n_estimators=300,
    learning_rate=0.05,
    random_state=42
)

ada.fit(X_train, y_train)

print("AdaBoost")
evaluate(ada, X_test, y_test)


AdaBoost
MAE : 74733.94375994339
RMSE: 10035892345.756239
R2  : 0.34416944486823486


In [16]:
MODEL_PATH = BASE_PATH / "models"

MODEL_PATH.mkdir(exist_ok=True)

print("Model folder created at:", MODEL_PATH)

Model folder created at: E:\C-DAC\Major Project\AI-Based-Maritime-Port-Intelligence\models


In [17]:
import joblib

joblib.dump(lgbm, MODEL_PATH / "lgbm_regression_model.pkl")

['E:\\C-DAC\\Major Project\\AI-Based-Maritime-Port-Intelligence\\models\\lgbm_regression_model.pkl']

In [18]:
import joblib
import pandas as pd
import numpy as np

lgbm_model = joblib.load(MODEL_PATH / "lgbm_regression_model.pkl")


In [19]:
new_input = {
    "ArrivalHour": 18,          # 6 PM
    "ArrivalDayOfWeek": 4,      # Friday (0=Mon)
    "BerthFeasible": 1,
    "BerthTime": 9
}

X_new = pd.DataFrame([new_input])

delay_minutes = lgbm_model.predict(X_new)[0]

print("Predicted Delay Minutes:", round(delay_minutes, 2))


Predicted Delay Minutes: 45121.99


In [20]:
scenarios = [
    # ArrivalHour, DayOfWeek, IsWeekend, BerthFeasible, BerthTime

    [16, 5,  1, 4],    # low delay (weekday morning, short berth time)
    [18, 4,  1, 9],    # medium delay (Friday evening)
    [20, 5,  0, 18],   # high delay (weekend, berth not feasible, long berth time)
]

X_batch = pd.DataFrame(scenarios, columns=FEATURES)

preds = lgbm_model.predict(X_batch)

for i, p in enumerate(preds):
    print(f"Scenario {i+1} Delay: {round(p, 2)} minutes")


Scenario 1 Delay: 37947.19 minutes
Scenario 2 Delay: 45121.99 minutes
Scenario 3 Delay: 52400.85 minutes


In [21]:
import joblib

joblib.dump(gbr, MODEL_PATH / "gbr_regression_model.pkl")

['E:\\C-DAC\\Major Project\\AI-Based-Maritime-Port-Intelligence\\models\\gbr_regression_model.pkl']

In [22]:
gbr_model = joblib.load(MODEL_PATH / "gbr_regression_model.pkl")

In [23]:
scenarios = [
    # ArrivalHour, DayOfWeek, IsWeekend, BerthFeasible, BerthTime

    [16, 5,  1, 4],    # low delay (weekday morning, short berth time)
    [18, 4,  1, 9],    # medium delay (Friday evening)
    [20, 5,  0, 18],   # high delay (weekend, berth not feasible, long berth time)
]

X_batch = pd.DataFrame(scenarios, columns=FEATURES)

preds = gbr_model.predict(X_batch)

for i, p in enumerate(preds):
    print(f"Scenario {i+1} Delay: {round(p, 2)} minutes")


Scenario 1 Delay: 9537.07 minutes
Scenario 2 Delay: 158713.75 minutes
Scenario 3 Delay: 80525.45 minutes
